In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
import warnings 
warnings.filterwarnings('ignore') 

In [2]:
df = pd.read_csv('../data/feature_engineered_data.csv')
display(df.head())
df.info()

,Unnamed: 0,reviewID,reviewerID,restaurantID,date,rating,reviewUsefulCount,reviewContent,flagged,name,...,capitalization_ratio,location_entropy,days_active,review_velocity,restaurant_rating_var,char_count,word_count,avg_word_length,has_friends,has_useful_votes
0,7267,dLS_UUea0Yh7et2YlGpUOw,hfQu0YNy_XW5oiiripgUFg,8d_DiHWB-pjBVW3a7D8EWg,2012-01-25,4,0,sunda amazing i heard many good things finally...,1,Mallory M.,...,0.0,0.0,24,0.041667,1.816092,447,72,6.208333,0,0
1,731,EDejgdY32F8Lr4ewh7FeiA,_jsZl-USMgrVVasNg50wAQ,boE4Ahsssqic7o5wQLI04w,2012-05-05,4,0,absolutely fantastic foodie community table gr...,0,Shawn K.,...,0.0,0.0,65,0.015385,0.812808,93,12,7.750000,0,0
2,9902,TRLtyGBT7VFODhxgyZ2tLA,sZxXpvmBUN2fSCtK_BZFoQ,ms5ge1XY9-Alu7HkybAMdQ,2007-11-09,3,0,i work right rarely go here they 5 personal st...,0,Dane K.,...,0.0,0.0,434,1.947005,1.163025,544,102,5.333333,1,1
3,25894,6QYZT3mkrmBmYNDUN18ILg,YMS9Fzy0OcOXFcS_qms_pg,3Mx2PM7v8qjqhgF3cDXxbQ,2011-06-20,3,0,this best big 3 brazilian steakhouses chicago ...,1,Pennywise S.,...,0.0,0.0,19,0.105263,1.238095,184,30,6.133333,1,0
4,17519,xuKTaBhJT0GaI4wRFZi6Qw,uvWWKett-BIRbvyLZXaloQ,vvhfPV-Llkd4fE2SHuLVvA,2009-09-28,4,0,i lunch the gage group 8 this first time there...,1,Rich C.,...,0.0,0.0,27,0.037037,1.090476,578,93,6.215054,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2481 entries, 0 to 2480
Data columns (total 38 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              2481 non-null   int64  
 1   reviewID                2481 non-null   object 
 2   reviewerID              2481 non-null   object 
 3   restaurantID            2481 non-null   object 
 4   date                    2481 non-null   object 
 5   rating                  2481 non-null   int64  
 6   reviewUsefulCount       2481 non-null   int64  
 7   reviewContent           2481 non-null   object 
 8   flagged                 2481 non-null   int64  
 9   name                    2481 non-null   object 
 10  location                2481 non-null   object 
 11  yelpJoinDate            2481 non-null   object 
 12  friendCount             2481 non-null   int64  
 13  reviewCount             2481 non-null   int64  
 14  firstCount              2481 non-null   

#
# 3.1 - Feature Pruning
#

In [3]:
import pandas as pd

# Load the engineered dataset from the end of Notebook 2
# (We use parse_dates just in case, though we will drop them shortly)
print("Original Shape:", df.shape)

# Define the list of string, ID, and raw date columns to remove
cols_to_drop = [
    'reviewID',       # Unique string - no predictive value
    'reviewerID',     # Unique string - (we extracted behavior from this already)
    'restaurantID',   # Unique string - (we extracted variance from this already)
    'date',           # Timestamp - (we extracted velocity from this already)
    'yelpJoinDate',   # Timestamp - (we extracted velocity from this already)
    'reviewContent',  # Raw text - (we extracted NLP features from this already)
    'name',           # String - Restaurant name
    'location'        # String - (we extracted entropy from this already)
]

# If we saved the 'review_year' or 'year_month' from our EDA phase, drop those too
for extra_col in ['review_year', 'year_month']:
    if extra_col in df.columns:
        cols_to_drop.append(extra_col)

# Create a new DataFrame for ML to keep our steps distinct
df_ml = df.drop(columns=cols_to_drop)

print("Shape after Pruning:", df_ml.shape)
print("\n--- Final ML Features ---")
# Check the datatypes to ensure absolutely everything left is an int or float
print(df_ml.dtypes)

Original Shape: (2481, 38)
Shape after Pruning: (2481, 29)

--- Final ML Features ---
Unnamed: 0                  int64
rating                      int64
reviewUsefulCount           int64
flagged                     int64
friendCount                 int64
reviewCount                 int64
firstCount                  int64
usefulCount                 int64
coolCount                   int64
funnyCount                  int64
complimentCount             int64
tipCount                    int64
fanCount                    int64
restaurantRating          float64
ReviewLength                int64
sentiment_score           float64
reviewer_sentiment_var    float64
lexical_diversity_ttr     float64
word_repetitiveness       float64
capitalization_ratio      float64
location_entropy          float64
days_active                 int64
review_velocity           float64
restaurant_rating_var     float64
char_count                  int64
word_count                  int64
avg_word_length           floa

#
# 3.2 Splitting Data
#

In [4]:
# sort strictly by time for chronological split 
# if in training dataset a reviewerid was classified as fake then test data will know for prev dates that in future it was fake thus 
# in past it must have been fake which is data leakage 

df = df.sort_values(by='date').reset_index(drop=True)

# calculate cutoff index for split
split_idx = int(len(df) * 0.8)

# split data sequentially
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print("Temporal Split Complete.")
print("Training reviews:", len(train_df))
print("Testing reviews:", len(test_df))

Temporal Split Complete.
Training reviews: 1984
Testing reviews: 497


In [9]:
# define columns to drop
cols_to_drop = [
    'reviewID', 'reviewerID', 'restaurantID', 'date', 
    'yelpJoinDate', 'reviewContent', 'name', 'location'
]

# prune non numerical columns from both sets
train_df = train_df.drop(columns=cols_to_drop, errors='ignore')
test_df = test_df.drop(columns=cols_to_drop, errors='ignore')

# separate features and target
X_train = train_df.drop(columns=['flagged'])
y_train = train_df['flagged']
X_test = test_df.drop(columns=['flagged'])
y_test = test_df['flagged']

# build correlation matrix
corr_matrix = X_train.corr().abs()

# select upper triangle of matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find features with correlation greater than threshold
to_drop_corr = [column for column in upper.columns if any(upper[column] > 0.95)]
print("Highly correlated features dropped:", to_drop_corr)

# drop redundant features from both sets
X_train = X_train.drop(columns=to_drop_corr)
X_test = X_test.drop(columns=to_drop_corr)

Highly correlated features dropped: ['coolCount', 'funnyCount', 'fanCount', 'word_repetitiveness', 'char_count', 'word_count']


In [10]:
df['flagged'].value_counts()

flagged
1    1269
0    1212
Name: count, dtype: int64

### What if data was imbalanced
Oversampling (SMOTE - Synthetic Minority Over-sampling Technique)
- What it is: SMOTE looks at the 5% of fake reviews, finds the mathematical midpoint between them (using K-Nearest Neighbors), and generates brand new, synthetic fake reviews until the classes are 50/50.

- Why use it: It balances the dataset without losing any of your precious genuine data.

- The Downside: It can sometimes create unrealistic "Frankenstein" data points, and it slows down training because you are inflating the size of your dataset.

Alternative A: Random Undersampling
- What it is: You randomly delete rows from the 95% Genuine class until it shrinks down to match the 5% Fake class.

- Why use it: It forces a 50/50 split and makes the dataset very small, meaning your model trains in seconds.

- The Downside: It is highly destructive. You are throwing away massive amounts of perfectly good, real data. It is rarely the best choice unless your dataset is impossibly huge.

Alternative B: Algorithmic Class Weights (The Production Standard)
- What it is: You do not touch or change the data at all. Instead, you modify the Machine Learning algorithm itself. You tell the algorithm: "If you guess a Genuine review incorrectly, I will penalize you 1 point.
- But if you guess a Fake review incorrectly, I will penalize you 20 points."

- Why use it: This is the safest, most professional method. You don't fake data (SMOTE) and you don't delete data (Undersampling).

- The Downside: Some older or simpler algorithms do not support class weights.

#
# 3.3 Saving Test and Train Data
#

In [8]:
from sklearn.preprocessing import StandardScaler

# initialize scaler
scaler = StandardScaler()

# fit and transform training data
X_train_scaled = scaler.fit_transform(X_train)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns)

# transform testing data only
X_test_scaled = scaler.transform(X_test)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# save to processed folder
X_train.to_csv('../data/X_train_scaled.csv', index=False)
X_test.to_csv('../data/X_test_scaled.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

